In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("job_salary_prediction_dataset.csv")
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [3]:
df.tail()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467
249999,Data Analyst,16,Diploma,2,Technology,Medium,UK,No,5,133084


In [4]:
df.isnull().sum()

job_title           0
experience_years    0
education_level     0
skills_count        0
industry            0
company_size        0
location            0
remote_work         0
certifications      0
salary              0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.shape

(250000, 10)

In [7]:
df.columns

Index(['job_title', 'experience_years', 'education_level', 'skills_count',
       'industry', 'company_size', 'location', 'remote_work', 'certifications',
       'salary'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   job_title         250000 non-null  object
 1   experience_years  250000 non-null  int64 
 2   education_level   250000 non-null  object
 3   skills_count      250000 non-null  int64 
 4   industry          250000 non-null  object
 5   company_size      250000 non-null  object
 6   location          250000 non-null  object
 7   remote_work       250000 non-null  object
 8   certifications    250000 non-null  int64 
 9   salary            250000 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 19.1+ MB


In [9]:
df.describe()

,experience_years,skills_count,certifications,salary
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,10.005408,9.997812,2.491928,145718.080524
std,6.060602,5.479288,1.706475,37407.952729
min,0.000000,1.000000,0.000000,31867.000000
25%,5.000000,5.000000,1.000000,119358.000000
50%,10.000000,10.000000,2.000000,143453.000000
75%,15.000000,15.000000,4.000000,169492.000000
max,20.000000,19.000000,5.000000,333046.000000


In [10]:
df['job_title'].unique()

array(['AI Engineer', 'Data Analyst', 'Frontend Developer',
       'Business Analyst', 'Product Manager', 'Backend Developer',
       'Machine Learning Engineer', 'DevOps Engineer',
       'Software Engineer', 'Cybersecurity Analyst', 'Data Scientist',
       'Cloud Engineer'], dtype=object)

In [11]:
# Ordinal Encoding
df['education_level'].unique()

array(['Bachelor', 'PhD', 'High School', 'Diploma', 'Master'],
      dtype=object)

In [12]:
df['industry'].unique()

array(['Healthcare', 'Telecom', 'Media', 'Retail', 'Manufacturing',
       'Education', 'Finance', 'Technology', 'Consulting', 'Government'],
      dtype=object)

In [13]:
## we can use Ordinal encoding
df['company_size'].unique()

array(['Medium', 'Small', 'Large', 'Enterprise', 'Startup'], dtype=object)

In [14]:
df['location'].unique()

array(['India', 'Australia', 'Singapore', 'Canada', 'Sweden', 'USA',
       'Netherlands', 'Remote', 'Germany', 'UK'], dtype=object)

In [15]:
# Label Encoding
df['remote_work'].unique()

array(['Hybrid', 'No', 'Yes'], dtype=object)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

In [17]:
x = df.drop(columns="salary")
y = df['salary']

In [18]:
xTrain, xTest, yTrain, yTest = train_test_split(
      x,
      y,
      test_size=0.3
)

In [19]:
xTrain.shape

(175000, 9)

In [20]:
xTest.shape

(75000, 9)

In [21]:
xTrain.columns

Index(['job_title', 'experience_years', 'education_level', 'skills_count',
       'industry', 'company_size', 'location', 'remote_work',
       'certifications'],
      dtype='object')

In [22]:
xTrain['company_size'].unique()

array(['Large', 'Small', 'Medium', 'Startup', 'Enterprise'], dtype=object)

In [23]:
xTrain['remote_work'].unique()

array(['Hybrid', 'Yes', 'No'], dtype=object)

In [24]:
pipeline = Pipeline([
      ("processor", ColumnTransformer(
      transformers=[
            ("numeric", StandardScaler(), [1, 3, 8]),
            ("ordinal endoer", OrdinalEncoder(
                  categories=[
                        ["High School", "Diploma", "Bachelor", "Master", 'PhD'],
                        ["Startup", "Small", "Medium", "Large", "Enterprise"],
                        ["Yes", "No", "Hybrid"]
                  ]
            ), [2, 5, 7]),
            ("OHE encoder", OneHotEncoder(drop='first', handle_unknown='ignore'), [0, 4, 6])
      ],
      remainder="passthrough"
))
])

In [25]:
xTrain = pipeline.fit_transform(xTrain)
xTest = pipeline.transform(xTest)

In [26]:
xTrain.shape

(175000, 35)

In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [28]:
import mlflow
from jobSalaryPrediction.utils.mlflow_config import configure_mlflow

d:\Job_Salary_Prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================
# EXPERIMENT 1: Classic ML Algorithms
# ============================================
print("=" * 60)
print("EXPERIMENT 1: Classic ML Algorithms")
print("=" * 60)


configure_mlflow("Experiment 1: Classic ML Algorithms")

models = {
      "Linear Regression": LinearRegression(),
      "RandomForestRegressor": RandomForestRegressor(max_depth=10, random_state=42),
      "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=10, random_state=42)
}


for model_name, model in models.items():
      with mlflow.start_run(run_name=model_name):
            mlflow.log_param("model", model_name)

            # Train the model
            model.fit(xTrain, yTrain)

            # Prediction
            yPred = model.predict(xTest)

            metrics = {
                  "r2": r2_score(yTest, yPred),
                  "mae": mean_absolute_error(yTest, yPred),
                  "mse": mean_squared_error(yTest, yPred)
            }
            for metric_name, value in metrics.items():
                  mlflow.log_metric(metric_name, value)

EXPERIMENT 1: Classic ML Algorithms
[2026-04-25 14:48:38,866: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as HimmatMagar

[2026-04-25 14:48:38,877: INFO: helpers: Accessing as HimmatMagar]
[2026-04-25 14:48:39,799: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/HimmatMagar/Job_Salary_Prediction "HTTP/1.1 200 OK"]
[2026-04-25 14:48:40,196: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "HimmatMagar/Job_Salary_Prediction"

[2026-04-25 14:48:40,214: INFO: helpers: Initialized MLflow to track repo "HimmatMagar/Job_Salary_Prediction"]


Repository HimmatMagar/Job_Salary_Prediction initialized!

[2026-04-25 14:48:40,214: INFO: helpers: Repository HimmatMagar/Job_Salary_Prediction initialized!]
Mlflow -> Dagshub | Experiment: Experiment 1: Classic ML Algorithms
🏃 View run Linear Regression at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0/runs/5285874ee8a6473fbf54e6c464c61d6a
🧪 View experiment at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0
🏃 View run RandomForestRegressor at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0/runs/7b179ad8b1a043c4852b6b976ac7fa20
🧪 View experiment at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0
🏃 View run DecisionTreeRegressor at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0/runs/caefcf4f3422467caf232002508d9ce8
🧪 View experiment at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/0


In [31]:
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
# ============================================
# EXPERIMENT 2: Ensemble Algorithms
# ============================================
print("=" * 60)
print("EXPERIMENT 2: Ensemble ML Algorithms")
print("=" * 60)


configure_mlflow("Experiment 2: Ensemble ML Algorithms")

models = {
      "Gradinet Boosing": GradientBoostingRegressor(max_depth=10, random_state=42),
      "XGBoost": XGBRegressor(random_state=42)
}


for model_name, model in models.items():
      with mlflow.start_run(run_name=model_name):
            mlflow.log_param("model", model_name)

            # Train the model
            model.fit(xTrain, yTrain)

            # Prediction
            yPred = model.predict(xTest)

            metrics = {
                  "r2": r2_score(yTest, yPred),
                  "mae": mean_absolute_error(yTest, yPred),
                  "mse": mean_squared_error(yTest, yPred)
            }
            for metric_name, value in metrics.items():
                  mlflow.log_metric(metric_name, value)

EXPERIMENT 2: Ensemble ML Algorithms


2026/04/25 15:09:43 INFO mlflow.tracking.fluent: Experiment with name 'Experiment 2: Ensemble ML Algorithms' does not exist. Creating a new experiment.


Mlflow -> Dagshub | Experiment: Experiment 2: Ensemble ML Algorithms
🏃 View run RandomForestRegressor at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/1/runs/956e5209646447a88c38772d09defe8e
🧪 View experiment at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/1
🏃 View run DecisionTreeRegressor at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/1/runs/6afe05a4f7894b4dbd6336c3eb3b4128
🧪 View experiment at: https://dagshub.com/HimmatMagar/Job_Salary_Prediction.mlflow/#/experiments/1


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# ============================================
# EXPERIMENT 3: Hyperparameter Tuning - Gradient Boosting
# ============================================
# print("=" * 60)
# print("EXPERIMENT 3: Hyperparameter Tuning - Gradient Boosting")
# print("=" * 60)

# configure_mlflow("Experiment 4: Hyperparameter Tuning - Gradient Boosting")

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(5, 15),
    'learning_rate': uniform(0.01, 0.3),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'subsample': uniform(0.5, 0.5)
}

# Base Gradient Boosting model
base_model = GradientBoostingRegressor(random_state=42)

# RandomizedSearchCV with 10 iterations
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=10,
    scoring='r2',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(xTrain, yTrain)
print(random_search.best_params_)
# with mlflow.start_run(run_name="GradientBoosting_Tuned"):
#       mlflow.log_params(random_search.best_params_)
#       for metric_name, value in metrics.items():
#             mlflow.log_metric(metric_name, value)
      
#       print(f"\nTest Set Metrics:")
#       print(f"  R²: {metrics['r2']:.4f}")
#       print(f"  MAE: {metrics['mae']:.4f}")
#       print(f"  MSE: {metrics['mse']:.4f}")